# 02 - Tabular Model

Trains CatBoost on `preprocessed.csv` and writes full-row `tabular_embeddings.csv` for fusion.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    print("CatBoost unavailable, using sklearn fallback:", exc)
    from sklearn.ensemble import HistGradientBoostingClassifier
    CATBOOST_AVAILABLE = False

DATA_PATH = "../data/preprocessed.csv"
OUT_PATH = "../data/tabular_embeddings.csv"

df = pd.read_csv(DATA_PATH).sort_values("claim_id").reset_index(drop=True)
target_col = "fraudfound_p"
feature_cols = [c for c in df.columns if c not in ["claim_id", target_col]]
X = df[feature_cols]
y = df[target_col].astype(int)
print(df.shape, len(feature_cols))

(15420, 53) 51


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

if CATBOOST_AVAILABLE:
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        eval_metric="AUC",
        loss_function="Logloss",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=100,
    )
    model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
    proba = model.predict_proba(X)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]
    raw_score = model.predict(X, prediction_type="RawFormulaVal").reshape(-1)
    leaf_indexes = model.calc_leaf_indexes(X)
else:
    model = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]
    raw_score = np.log(np.clip(proba, 1e-6, 1 - 1e-6) / np.clip(1 - proba, 1e-6, 1))
    leaf_indexes = np.tile(proba.reshape(-1, 1), (1, 16))

print("ROC-AUC:", roc_auc_score(y_test, test_proba))
print("PR-AUC:", average_precision_score(y_test, test_proba))
print(classification_report(y_test, (test_proba >= 0.5).astype(int), zero_division=0))

0:	test: 0.7754846	best: 0.7754846 (0)	total: 13.6ms	remaining: 6.79s
100:	test: 0.8303907	best: 0.8307208 (97)	total: 662ms	remaining: 2.61s
200:	test: 0.8306536	best: 0.8330384 (195)	total: 1.25s	remaining: 1.85s
300:	test: 0.8269655	best: 0.8330384 (195)	total: 1.86s	remaining: 1.23s
400:	test: 0.8241574	best: 0.8330384 (195)	total: 2.51s	remaining: 620ms
499:	test: 0.8215582	best: 0.8330384 (195)	total: 3.14s	remaining: 0us

bestTest = 0.8330384196
bestIteration = 195

Shrink model to first 196 iterations.
ROC-AUC: 0.8330384195855046
PR-AUC: 0.2191679125302241
              precision    recall  f1-score   support

           0       0.98      0.71      0.82      2899
           1       0.15      0.79      0.25       185

    accuracy                           0.71      3084
   macro avg       0.56      0.75      0.54      3084
weighted avg       0.93      0.71      0.79      3084



In [7]:
# Convert model internals into a compact fixed-width tabular embedding.
embedding_dim = 16
if leaf_indexes.shape[1] >= embedding_dim:
    positions = np.linspace(0, leaf_indexes.shape[1] - 1, embedding_dim).round().astype(int)
    emb = leaf_indexes[:, positions].astype(float)
else:
    emb = np.zeros((len(df), embedding_dim), dtype=float)
    emb[:, :leaf_indexes.shape[1]] = leaf_indexes.astype(float)

# Normalize each embedding column for fusion stability.
emb = (emb - emb.mean(axis=0)) / (emb.std(axis=0) + 1e-6)

out = pd.DataFrame({"claim_id": df["claim_id"].astype(int)})
for i in range(embedding_dim):
    out[f"tab_emb_{i:03d}"] = emb[:, i]
out["tabular_raw_score"] = raw_score
out["tabular_fraud_probability"] = proba
out[target_col] = y.values

assert len(out) == len(df)
assert out["claim_id"].is_unique
out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, out.shape)
out.head()

Saved: ../data/tabular_embeddings.csv (15420, 20)


,claim_id,tab_emb_000,tab_emb_001,tab_emb_002,tab_emb_003,tab_emb_004,tab_emb_005,tab_emb_006,tab_emb_007,tab_emb_008,tab_emb_009,tab_emb_010,tab_emb_011,tab_emb_012,tab_emb_013,tab_emb_014,tab_emb_015,tabular_raw_score,tabular_fraud_probability,fraudfound_p
0,1,-5.661843,-1.624781,-0.178685,-0.871563,-0.109236,0.056308,0.927970,-1.003001,-0.287508,-0.602661,-1.593754,-0.073974,0.223717,-0.37566,0.278772,-0.057296,-1.659573,0.159819,0
1,2,1.626046,-0.092651,0.849183,-0.871563,-0.109236,-0.396656,-0.959237,-0.568405,1.257655,-1.037202,-0.693502,-0.073974,-0.720910,-0.37566,0.278772,-1.322669,0.242665,0.560370,0
2,3,1.626046,-0.625566,-1.206553,1.130326,1.705570,-1.529066,-0.015633,2.101255,0.544503,2.004585,0.822711,0.616442,2.112972,-0.37566,1.277353,-0.222345,-0.540218,0.368137,0
3,4,2.054745,1.439479,0.849183,1.630799,1.478719,-1.585687,-1.292273,-1.189256,0.306786,1.895949,1.391291,0.616442,0.932188,2.41006,-0.608855,0.767947,-4.048571,0.017148,0
4,5,1.911845,1.506093,-1.720487,-0.371090,-1.016639,-1.076102,-0.071139,-1.251341,0.663361,-1.037202,-1.546372,-0.189044,0.223717,-0.37566,0.278772,0.107753,-2.796095,0.057536,0
